# InsightFuel Analytics — 02. Silver

Transformação da camada Bronze para a camada Silver.

**Entradas**
- `bronze.anp_raw`
- `bronze.populacao_raw`
- `bronze.pib_raw`

**Saídas**
- `silver.combustivel_tratado`
- `silver.municipio_tratado`
- `silver.indicadores_municipio_tratado`

Nesta etapa fazemos padronização, limpeza e remoção de duplicidades.


## 1. Configuração

In [2]:
from pathlib import Path
import re
import unicodedata
import pandas as pd
import duckdb

def find_project_root(start=Path.cwd()):
    start = Path(start).resolve()
    for p in [start] + list(start.parents):
        if (p / "data" / "bronze").exists():
            return p
    raise FileNotFoundError("Pasta data/bronze não encontrada.")

PROJECT_ROOT = find_project_root()
DB_PATH = PROJECT_ROOT / "database" / "insightfuel.duckdb"

if not DB_PATH.exists():
    raise FileNotFoundError(
        "DuckDB não encontrado. Execute primeiro o notebook 01_ingestao.ipynb."
    )

con = duckdb.connect(str(DB_PATH))

print("Projeto:", PROJECT_ROOT)
print("Banco:", DB_PATH)


Projeto: C:\Users\Bruna\anaconda_projects\Trabalho_Armazenamento
Banco: C:\Users\Bruna\anaconda_projects\Trabalho_Armazenamento\database\insightfuel.duckdb


## 2. Verificar Bronze

In [3]:
tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'bronze'
    ORDER BY table_name
""").df()

display(tables)

required = {"anp_raw", "populacao_raw", "pib_raw"}
missing = required - set(tables["table_name"])
assert not missing, f"Tabelas ausentes: {missing}"

for table in sorted(required):
    print(f"\n--- bronze.{table} ---")
    display(con.execute(f"DESCRIBE bronze.{table}").df())


,table_name
0,anp_raw
1,ingestion_log
2,pib_raw
3,populacao_raw



--- bronze.anp_raw ---


,column_name,column_type,null,key,default,extra
0,Regiao - Sigla,VARCHAR,YES,None,None,None
1,Estado - Sigla,VARCHAR,YES,None,None,None
2,Municipio,VARCHAR,YES,None,None,None
3,Revenda,VARCHAR,YES,None,None,None
4,CNPJ da Revenda,VARCHAR,YES,None,None,None
5,Nome da Rua,VARCHAR,YES,None,None,None
6,Numero Rua,VARCHAR,YES,None,None,None
7,Complemento,VARCHAR,YES,None,None,None
8,Bairro,VARCHAR,YES,None,None,None
9,Cep,VARCHAR,YES,None,None,None



--- bronze.pib_raw ---


,column_name,column_type,null,key,default,extra
0,Ano,BIGINT,YES,None,None,None
1,Código da Grande Região,BIGINT,YES,None,None,None
2,Nome da Grande Região,VARCHAR,YES,None,None,None
3,Código da Unidade da Federação,BIGINT,YES,None,None,None
4,Sigla da Unidade da Federação,VARCHAR,YES,None,None,None
5,Nome da Unidade da Federação,VARCHAR,YES,None,None,None
6,Código do Município,BIGINT,YES,None,None,None
7,Nome do Município,VARCHAR,YES,None,None,None
8,Região Metropolitana,VARCHAR,YES,None,None,None
9,Código da Mesorregião,BIGINT,YES,None,None,None



--- bronze.populacao_raw ---


,column_name,column_type,null,key,default,extra
0,UF,VARCHAR,YES,None,None,None
1,COD. UF,DOUBLE,YES,None,None,None
2,COD. MUNIC,DOUBLE,YES,None,None,None
3,NOME DO MUNICÍPIO,VARCHAR,YES,None,None,None
4,POPULAÇÃO APURADA IBGE \n- CENSO DEMOGRÁFICO 2...,VARCHAR,YES,None,None,None


## 3. Funções auxiliares

In [4]:
def normalize_name(value):
    value = str(value).strip().lower()
    value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", "_", value).strip("_")

def normalize_columns(df):
    df = df.copy()
    used = {}
    cols = []
    for col in df.columns:
        base = normalize_name(col)
        n = used.get(base, 0)
        cols.append(base if n == 0 else f"{base}_{n}")
        used[base] = n + 1
    df.columns = cols
    return df

def clean_text(df):
    df = df.copy()
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = (
            df[col].astype("string").str.strip()
            .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        )
    return df

def remove_exact_duplicates(df):
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    return df, before - len(df)

def find_column(columns, aliases):
    normalized = {normalize_name(c): c for c in columns}
    for alias in aliases:
        if normalize_name(alias) in normalized:
            return normalized[normalize_name(alias)]
    return None


# 4. ANP → `silver.combustivel_tratado`

In [5]:
anp = con.execute("SELECT * FROM bronze.anp_raw").df()

anp = normalize_columns(anp)
anp = clean_text(anp)
anp, removed = remove_exact_duplicates(anp)

print("Linhas:", f"{len(anp):,}")
print("Colunas:", len(anp.columns))
print("Duplicidades exatas removidas:", removed)

display(anp.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Linhas: 2,619,039
Colunas: 19
Duplicidades exatas removidas: 0


,regiao_sigla,estado_sigla,municipio,revenda,cnpj_da_revenda,nome_da_rua,numero_rua,complemento,bairro,cep,produto,data_da_coleta,valor_de_venda,valor_de_compra,unidade_de_medida,bandeira,arquivo_origem,ano_arquivo,semestre_arquivo
0,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,<NA>,FEITORIA,93048-000,GASOLINA,01/01/2021,"4,599",NaN,R$ / litro,BRANCA,anp_2021_1_raw.csv,2021,1
1,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,<NA>,FEITORIA,93048-000,ETANOL,01/01/2021,"4,199",NaN,R$ / litro,BRANCA,anp_2021_1_raw.csv,2021,1
2,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,<NA>,FEITORIA,93048-000,GASOLINA ADITIVADA,01/01/2021,"4,799",NaN,R$ / litro,BRANCA,anp_2021_1_raw.csv,2021,1
3,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,<NA>,FEITORIA,93048-000,DIESEL,01/01/2021,"3,499",NaN,R$ / litro,BRANCA,anp_2021_1_raw.csv,2021,1
4,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,<NA>,FEITORIA,93048-000,DIESEL S10,01/01/2021,"3,599",NaN,R$ / litro,BRANCA,anp_2021_1_raw.csv,2021,1


### 4.1. Conversão de campos de data/período

In [6]:
date_candidates = [
    c for c in anp.columns
    if any(x in c for x in ["data", "date", "periodo", "competencia", "mes_ano"])
]

print("Candidatas:", date_candidates)

for col in date_candidates:
    if anp[col].dtype == "object" or str(anp[col].dtype).startswith("string"):
        converted = pd.to_datetime(anp[col], errors="coerce", dayfirst=True)
        non_null = anp[col].notna().sum()
        if non_null and converted.notna().sum() / non_null >= 0.80:
            anp[col] = converted
            print("Convertida:", col)


Candidatas: ['data_da_coleta']
Convertida: data_da_coleta


### 4.2. Conversão de campos de preço/valor

In [7]:
value_candidates = [
    c for c in anp.columns
    if "preco" in c or "valor" in c
]

print("Candidatas:", value_candidates)

for col in value_candidates:
    if anp[col].dtype == "object" or str(anp[col].dtype).startswith("string"):
        cleaned = (
            anp[col].astype("string")
            .str.replace(".", "", regex=False)
            .str.replace(",", ".", regex=False)
        )
        converted = pd.to_numeric(cleaned, errors="coerce")
        non_null = anp[col].notna().sum()
        if non_null and converted.notna().sum() / non_null >= 0.80:
            anp[col] = converted
            print("Convertida:", col)


Candidatas: ['valor_de_venda', 'valor_de_compra']
Convertida: valor_de_venda


# 5. População → `silver.municipio_tratado`

In [8]:
pop = con.execute("SELECT * FROM bronze.populacao_raw").df()

pop = normalize_columns(pop)
pop = clean_text(pop)
pop, removed = remove_exact_duplicates(pop)

print("Linhas:", f"{len(pop):,}")
print("Colunas:", len(pop.columns))
print("Duplicidades exatas removidas:", removed)

display(pop.head())
print("Colunas:", list(pop.columns))


Linhas: 5,602
Colunas: 5
Duplicidades exatas removidas: 1


,uf,cod_uf,cod_munic,nome_do_municipio,populacao_apurada_ibge_censo_demografico_2022_e_malha_territorial_2023
0,RO,11.0,15.0,Alta Floresta D'Oeste,21494
1,RO,11.0,23.0,Ariquemes,96833
2,RO,11.0,31.0,Cabixi,5351
3,RO,11.0,49.0,Cacoal,86887
4,RO,11.0,56.0,Cerejeiras,15890


Colunas: ['uf', 'cod_uf', 'cod_munic', 'nome_do_municipio', 'populacao_apurada_ibge_censo_demografico_2022_e_malha_territorial_2023']


In [16]:
ibge_aliases = [
    "cod_munic", "codigo_ibge", 'codigo_do_municipio', "cod_ibge",
]

pop_ibge = find_column(pop.columns, ibge_aliases)
print("Código IBGE:", pop_ibge)

if pop_ibge:
    pop["codigo_ibge"] = (
        pop[pop_ibge].astype("string")
        .str.extract(r"(\d+)", expand=False)
        .str.zfill(7)
    )
    print("Cobertura:", f"{pop['codigo_ibge'].notna().mean():.1%}")
    display(pop[["codigo_ibge"]].drop_duplicates().head(10))
else:
    print("⚠ Código IBGE não identificado automaticamente.")


Código IBGE: cod_munic
Cobertura: 99.4%


,codigo_ibge
0,0000015
1,0000023
2,0000031
3,0000049
4,0000056
5,0000064
6,0000072
7,0000080
8,0000098
9,0000106


# 6. PIB → `silver.indicadores_municipio_tratado`

In [17]:
pib = con.execute("SELECT * FROM bronze.pib_raw").df()

pib = normalize_columns(pib)
pib = clean_text(pib)
pib, removed = remove_exact_duplicates(pib)

print("Linhas:", f"{len(pib):,}")
print("Colunas:", len(pib.columns))
print("Duplicidades exatas removidas:", removed)

display(pib.head())
print("Colunas:", list(pib.columns))


Linhas: 77,965
Colunas: 43
Duplicidades exatas removidas: 0


,ano,codigo_da_grande_regiao,nome_da_grande_regiao,codigo_da_unidade_da_federacao,sigla_da_unidade_da_federacao,nome_da_unidade_da_federacao,codigo_do_municipio,nome_do_municipio,regiao_metropolitana,codigo_da_mesorregiao,...,valor_adicionado_bruto_da_industria_a_precos_correntes_r_1_000,valor_adicionado_bruto_dos_servicos_a_precos_correntes_exceto_administracao_defesa_educacao_e_saude_publicas_e_seguridade_social_r_1_000,valor_adicionado_bruto_da_administracao_defesa_educacao_e_saude_publicas_e_seguridade_social_a_precos_correntes_r_1_000,valor_adicionado_bruto_total_a_precos_correntes_r_1_000,impostos_liquidos_de_subsidios_sobre_produtos_a_precos_correntes_r_1_000,produto_interno_bruto_a_precos_correntes_r_1_000,produto_interno_bruto_per_capita_a_precos_correntes_r_1_00,atividade_com_maior_valor_adicionado_bruto,atividade_com_segundo_maior_valor_adicionado_bruto,atividade_com_terceiro_maior_valor_adicionado_bruto
0,2010,1,Norte,11,RO,Rondônia,1100015,Alta Floresta D'Oeste,<NA>,1102,...,16118.534,62496.185,93244.656,241119.767,20957.111,262076.878,10731.18,"Administração, defesa, educação e saúde públic...","Pecuária, inclusive apoio à pecuária",Demais serviços
1,2010,1,Norte,11,RO,Rondônia,1100023,Ariquemes,<NA>,1102,...,287138.585,494946.267,343867.731,1199664.227,165029.553,1364693.780,15103.86,"Administração, defesa, educação e saúde públic...",Demais serviços,Comércio e reparação de veículos automotores e...
2,2010,1,Norte,11,RO,Rondônia,1100031,Cabixi,<NA>,1102,...,3252.506,12677.210,25170.235,65400.772,4210.342,69611.114,11033.62,"Administração, defesa, educação e saúde públic...","Pecuária, inclusive apoio à pecuária",Demais serviços
3,2010,1,Norte,11,RO,Rondônia,1100049,Cacoal,<NA>,1102,...,182051.537,465447.325,298454.309,1041212.374,145281.717,1186494.091,15095.15,"Administração, defesa, educação e saúde públic...",Demais serviços,Comércio e reparação de veículos automotores e...
4,2010,1,Norte,11,RO,Rondônia,1100056,Cerejeiras,<NA>,1102,...,19734.484,80724.991,63018.270,192454.160,29567.029,222021.189,13037.06,"Administração, defesa, educação e saúde públic...",Demais serviços,Comércio e reparação de veículos automotores e...


Colunas: ['ano', 'codigo_da_grande_regiao', 'nome_da_grande_regiao', 'codigo_da_unidade_da_federacao', 'sigla_da_unidade_da_federacao', 'nome_da_unidade_da_federacao', 'codigo_do_municipio', 'nome_do_municipio', 'regiao_metropolitana', 'codigo_da_mesorregiao', 'nome_da_mesorregiao', 'codigo_da_microrregiao', 'nome_da_microrregiao', 'codigo_da_regiao_geografica_imediata', 'nome_da_regiao_geografica_imediata', 'municipio_da_regiao_geografica_imediata', 'codigo_da_regiao_geografica_intermediaria', 'nome_da_regiao_geografica_intermediaria', 'municipio_da_regiao_geografica_intermediaria', 'codigo_concentracao_urbana', 'nome_concentracao_urbana', 'tipo_concentracao_urbana', 'codigo_arranjo_populacional', 'nome_arranjo_populacional', 'hierarquia_urbana', 'hierarquia_urbana_principais_categorias', 'codigo_da_regiao_rural', 'nome_da_regiao_rural', 'regiao_rural_segundo_classificacao_do_nucleo', 'amazonia_legal', 'semiarido', 'cidade_regiao_de_sao_paulo', 'valor_adicionado_bruto_da_agropecuaria_

In [18]:
pib_ibge = find_column(pib.columns, ibge_aliases)
year_aliases = ["ano", "ano_referencia", "ano_de_referencia", "periodo"]
pib_year = find_column(pib.columns, year_aliases)

print("Código IBGE:", pib_ibge)
print("Ano:", pib_year)

if pib_ibge:
    pib["codigo_ibge"] = (
        pib[pib_ibge].astype("string")
        .str.extract(r"(\d+)", expand=False)
        .str.zfill(7)
    )

if pib_year:
    pib["ano"] = pd.to_numeric(
        pib[pib_year].astype("string").str.extract(r"(\d{4})", expand=False),
        errors="coerce"
    )

display(pib.head())


Código IBGE: codigo_do_municipio
Ano: ano


,ano,codigo_da_grande_regiao,nome_da_grande_regiao,codigo_da_unidade_da_federacao,sigla_da_unidade_da_federacao,nome_da_unidade_da_federacao,codigo_do_municipio,nome_do_municipio,regiao_metropolitana,codigo_da_mesorregiao,...,valor_adicionado_bruto_dos_servicos_a_precos_correntes_exceto_administracao_defesa_educacao_e_saude_publicas_e_seguridade_social_r_1_000,valor_adicionado_bruto_da_administracao_defesa_educacao_e_saude_publicas_e_seguridade_social_a_precos_correntes_r_1_000,valor_adicionado_bruto_total_a_precos_correntes_r_1_000,impostos_liquidos_de_subsidios_sobre_produtos_a_precos_correntes_r_1_000,produto_interno_bruto_a_precos_correntes_r_1_000,produto_interno_bruto_per_capita_a_precos_correntes_r_1_00,atividade_com_maior_valor_adicionado_bruto,atividade_com_segundo_maior_valor_adicionado_bruto,atividade_com_terceiro_maior_valor_adicionado_bruto,codigo_ibge
0,2010,1,Norte,11,RO,Rondônia,1100015,Alta Floresta D'Oeste,<NA>,1102,...,62496.185,93244.656,241119.767,20957.111,262076.878,10731.18,"Administração, defesa, educação e saúde públic...","Pecuária, inclusive apoio à pecuária",Demais serviços,1100015
1,2010,1,Norte,11,RO,Rondônia,1100023,Ariquemes,<NA>,1102,...,494946.267,343867.731,1199664.227,165029.553,1364693.780,15103.86,"Administração, defesa, educação e saúde públic...",Demais serviços,Comércio e reparação de veículos automotores e...,1100023
2,2010,1,Norte,11,RO,Rondônia,1100031,Cabixi,<NA>,1102,...,12677.210,25170.235,65400.772,4210.342,69611.114,11033.62,"Administração, defesa, educação e saúde públic...","Pecuária, inclusive apoio à pecuária",Demais serviços,1100031
3,2010,1,Norte,11,RO,Rondônia,1100049,Cacoal,<NA>,1102,...,465447.325,298454.309,1041212.374,145281.717,1186494.091,15095.15,"Administração, defesa, educação e saúde públic...",Demais serviços,Comércio e reparação de veículos automotores e...,1100049
4,2010,1,Norte,11,RO,Rondônia,1100056,Cerejeiras,<NA>,1102,...,80724.991,63018.270,192454.160,29567.029,222021.189,13037.06,"Administração, defesa, educação e saúde públic...",Demais serviços,Comércio e reparação de veículos automotores e...,1100056


## 7. Gravar as tabelas Silver

In [23]:
con.execute("CREATE SCHEMA IF NOT EXISTS silver")

con.register("tmp_anp", anp)
con.execute("""
    CREATE OR REPLACE TABLE silver.combustivel_tratado AS
    SELECT * FROM tmp_anp
""")

con.register("tmp_pop", pop)
con.execute("""
    CREATE OR REPLACE TABLE silver.municipio_tratado AS
    SELECT * FROM tmp_pop
""")

con.register("tmp_pib", pib)
con.execute("""
    CREATE OR REPLACE TABLE silver.indicadores_municipio_tratado AS
    SELECT * FROM tmp_pib
""")

print("✓ Silver gravada no DuckDB.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Silver gravada no DuckDB.


## 8. Testes finais

In [24]:
silver_tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'silver'
    ORDER BY table_name
""").df()

display(silver_tables)

expected = {
    "combustivel_tratado",
    "municipio_tratado",
    "indicadores_municipio_tratado"
}

missing = expected - set(silver_tables["table_name"])
assert not missing, f"Tabelas Silver ausentes: {missing}"

counts = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM silver.combustivel_tratado) AS combustivel,
        (SELECT COUNT(*) FROM silver.municipio_tratado) AS municipio,
        (SELECT COUNT(*) FROM silver.indicadores_municipio_tratado) AS indicadores
""").df()

display(counts)
assert (counts.iloc[0] > 0).all()

print("✓ Testes finais concluídos.")
con.close()
print("Conexão com o DuckDB encerrada.")


,table_name
0,combustivel_tratado
1,indicadores_municipio_tratado
2,municipio_tratado


,combustivel,municipio,indicadores
0,2619039,5602,77965


✓ Testes finais concluídos.
Conexão com o DuckDB encerrada.
